In [1]:
import pandas as pd
import numpy as np 
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


In [2]:
# PREDICTORS = ["pH", "Cond", "Temp", "OD", "Tds", "Resist", "Salin", "ORP", "IP", "Cor"] # 10 entradas
PREDICTORS = ["pH", "Cond", "Temp", "OD", "Tds", "ORP"] # 6 entradas

TARGETS = ["Fe", "Al", "As", "Pb", "Zn", "Hg", "Co", "V", "Ba", "Mn"] # 10 saidas

SCALER = StandardScaler()
OUT_SCALER = StandardScaler()

In [3]:
Dataset = pd.read_excel("../Dados/Dados.xlsx")

Datasets = []
NormDatasets = []


for n in range(1, 5):
    n_data = Dataset[ Dataset["Pontos"] == f"P{n}" ]
    n_data = n_data.drop(columns=["Campanhas", "Pontos"])
    
    n_data_norm = n_data
    
    n_data_norm[PREDICTORS] = SCALER.fit_transform(n_data[PREDICTORS])
    n_data_norm[TARGETS] = OUT_SCALER.fit_transform(n_data[TARGETS])
    
    NormDatasets.append(n_data)
    Datasets.append(n_data)

O potencial de redução da nova base deve ser avaliado de
forma a diminuir ao máximo a dimensionalidade, sem perda
significativa das informações. No entanto, esta é uma questão
ainda em aberto (Abdi et al. 2010)

A seguir, algumas das regras utilizadas para definir a redução
de C.P. apresentadas por Savegnago et al. (2011):

    • E.V.P.A. acima de 80%, ou algum valor pré-definido.
    Este valor pode variar de acordo com a referência
    utilizada;

    • Selecionar C.P. com autovalores maiores que a média
    de autovalores.

In [4]:
limiar_evpa = 0.9  

for i, Dataset in enumerate(NormDatasets):
    print(f"\n+++++++++++ Pontos {i} +++++++++++++++")

    X = Dataset[PREDICTORS]

    # Ajusta PCA com todas as componentes possíveis
    pca = PCA()
    pca.fit(X)

    # =========================
    # Critério 1: EVPA ≥ 80%
    # =========================
    evpa_acumulada = np.cumsum(pca.explained_variance_ratio_)
    n_evpa = np.argmax(evpa_acumulada >= limiar_evpa) + 1

    # =========================
    # Critério 2: Autovalores > média
    # =========================
    autovalores = pca.explained_variance_
    media_autovalores = np.mean(autovalores)
    n_autovalores = np.sum(autovalores > media_autovalores)

    # =========================
    # Impressão dos resultados
    # =========================
    print(f"Variância explicada (%): {np.round(pca.explained_variance_ratio_ * 100, 3)}")
    print(f"Variância acumulada (%): {np.round(evpa_acumulada * 100, 3)}")

    print(f"\n→ Critério EVPA ≥ 80%:")
    print(f"  Número de CPs: {n_evpa}")
    print(f"  EVPA atingida (%): {evpa_acumulada[n_evpa-1]*100:.2f}")

    print(f"\n→ Critério Autovalores > média:")
    print(f"  Média dos autovalores: {media_autovalores:.4f}")
    print(f"  Número de CPs: {n_autovalores}")



+++++++++++ Pontos 0 +++++++++++++++
Variância explicada (%): [4.7499e+01 2.1657e+01 1.3197e+01 1.1960e+01 5.6850e+00 2.0000e-03]
Variância acumulada (%): [ 47.499  69.156  82.353  94.313  99.998 100.   ]

→ Critério EVPA ≥ 80%:
  Número de CPs: 4
  EVPA atingida (%): 94.31

→ Critério Autovalores > média:
  Média dos autovalores: 1.0400
  Número de CPs: 2

+++++++++++ Pontos 1 +++++++++++++++
Variância explicada (%): [4.0906e+01 2.2859e+01 1.4603e+01 1.1660e+01 9.9710e+00 1.0000e-03]
Variância acumulada (%): [ 40.906  63.765  78.368  90.028  99.999 100.   ]

→ Critério EVPA ≥ 80%:
  Número de CPs: 4
  EVPA atingida (%): 90.03

→ Critério Autovalores > média:
  Média dos autovalores: 1.0400
  Número de CPs: 2

+++++++++++ Pontos 2 +++++++++++++++
Variância explicada (%): [41.469 22.764 16.751 10.735  8.282  0.   ]
Variância acumulada (%): [ 41.469  64.233  80.983  91.718 100.    100.   ]

→ Critério EVPA ≥ 80%:
  Número de CPs: 4
  EVPA atingida (%): 91.72

→ Critério Autovalores > mé